# X (Twitter) Scraper

Test all X scraper endpoints (`client.scrape.x`):
- **Posts — collect by URL**
- **Posts — discover by profile URL** (optional date range)
- **Posts — discover by profiles array** (many profiles in one request)
- **Profiles — collect by URL** (cap posts via `max_number_of_posts`)
- **Profiles — discover by user name**

Two datasets back this scraper: posts (`gd_lwxkxvnf1cynvib9co`) and profiles (`gd_lwxmeb2u1cniijd7t4`).

---

## Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

API_TOKEN = os.getenv("BRIGHTDATA_API_TOKEN")
if not API_TOKEN:
    raise ValueError("Set BRIGHTDATA_API_TOKEN in .env file")

print(f"API Token: {API_TOKEN[:10]}...{API_TOKEN[-4:]}")

# Check SDK version and location
import brightdata
print(f"SDK Version: {brightdata.__version__}")
print(f"SDK Location: {brightdata.__file__}")
print("Setup complete!")

API Token: 02f48d4c-2...9a61
SDK Version: 2.3.2
SDK Location: /Users/ns/Desktop/projects/sdk-python/src/brightdata/__init__.py
Setup complete!


## Initialize Client

In [2]:
from brightdata import BrightDataClient

client = BrightDataClient(token=API_TOKEN)

print("Client initialized")

Client initialized


---
## Test 1: Posts — Collect by URL (single)

Scrape a single X post by its status URL.

In [3]:
POST_URL = "https://x.com/FabrizioRomano/status/1683559267524136962"

print(f"Scraping post: {POST_URL}\n")

async with client:
    result = await client.scrape.x.posts(url=POST_URL)

print(f"Success: {result.success}")
print(f"Status: {result.status}")

if result.success and result.data:
    data = result.data
    print(f"\n=== Post Data ({len(data)} fields) ===")
    for key, value in data.items():
        val_str = str(value)[:80] + "..." if len(str(value)) > 80 else str(value)
        print(f"  {key}: {val_str}")
else:
    print(f"\nError: {result.error}")

Scraping post: https://x.com/FabrizioRomano/status/1683559267524136962

Success: True
Status: ready

=== Post Data (33 fields) ===
  id: 1683559267524136962
  user_posted: FabrizioRomano
  name: Fabrizio Romano
  description: Here we go? 😵‍💫🏀🇸🇦
  date_posted: 2023-07-24T19:26:23.000Z
  photos: None
  url: https://x.com/FabrizioRomano/status/1683559267524136962
  quoted_post: {'photos': None, 'videos': None}
  tagged_users: None
  replies: 2569
  reposts: 13133
  likes: 244053
  views: 19868955
  external_url: None
  hashtags: None
  followers: 27933860
  biography: Here we go! © fabrizio.romano93@gmail.com 📩
  posts_count: 71978
  profile_image_link: https://pbs.twimg.com/profile_images/1741753635158024192/j0m8Ucvv_normal.jpg
  following: 2668
  is_verified: True
  quotes: 606
  bookmarks: 675
  parent_post_details: {'post_id': '1683559267524136962', 'profile_id': '330262748', 'profile_name': 'F...
  external_image_urls: None
  videos: None
  external_video_urls: None
  verification_ty

---
## Test 2: Posts — Collect by URL (batch)

Scrape several posts at once by passing a list of URLs.

In [5]:
POST_URLS = [
    "https://x.com/FabrizioRomano/status/1552015619251634176",
    "https://x.com/CNN/status/1796673270344810776",
]

print(f"Batch scraping {len(POST_URLS)} posts...")

async with client:
    result = await client.scrape.x.posts(url=POST_URLS)

# A batch call returns a single ScrapeResult whose .data holds all the posts
# (one record per input URL), not a list of ScrapeResult objects.
print(f"Success: {result.success} | Status: {result.status}")

if result.success and result.data:
    posts = result.data if isinstance(result.data, list) else [result.data]
    print(f"Got {len(posts)} post(s):")
    print()
    for i, post in enumerate(posts):
        print(f"=== Post {i+1} ===")
        print(f"  User: {post.get('user_posted', 'N/A')}")
        print(f"  Text: {str(post.get('description', post.get('text', 'N/A')))[:80]}")
        print(f"  URL:  {post.get('url', 'N/A')}")
        print()
else:
    print(f"Error: {result.error}")

Batch scraping 2 posts...
Success: True | Status: ready
Got 2 post(s):

=== Post 1 ===
  User: CNN
  Text: Marian Robinson, the mother of former first lady Michelle Obama, has died, accor
  URL:  https://x.com/CNN/status/1796673270344810776

=== Post 2 ===
  User: FabrizioRomano
  Text: …and this is how AS Roma unveiled Paulo Dybala as new signing tonight, in front 
  URL:  https://x.com/FabrizioRomano/status/1552015619251634176



---
## Test 3: Posts — Discover by Profile URL

Discover posts from a profile, optionally within a date range. The same range
applies to all URLs. Leave the dates out to discover without a window.

In [ ]:
PROFILE_URL = "https://x.com/elonmusk"

print(f"Discovering posts from: {PROFILE_URL}")

async with client:
    result = await client.scrape.x.posts_by_profile(
        url=PROFILE_URL,
        start_date="2024-01-01",
        end_date="2024-03-15",
        limit_per_input=2,  # cap posts per profile (small/fast while testing)
    )

print(f"Success: {result.success} | Status: {result.status}")

if result.success and result.data:
    posts = result.data if isinstance(result.data, list) else [result.data]
    print(f"Discovered {len(posts)} post(s).")
    print()
    for i, p in enumerate(posts):
        # Many posts have no caption (media-only / reposts) -> description is None.
        text = p.get("description") or "(no caption - media/repost)"
        has_media = bool(p.get("photos") or p.get("videos"))
        print(f"=== Post {i+1} ===")
        print(f"  Date:  {p.get('date_posted', 'N/A')}")
        print(f"  Text:  {str(text)[:80]}")
        print(f"  Likes: {p.get('likes', 'N/A')} | Reposts: {p.get('reposts', 'N/A')}")
        print(f"  Media: {'yes' if has_media else 'no'}")
        print(f"  URL:   {p.get('url', 'N/A')}")
        print()

    # Full field dump of the first record, to see everything the API returns
    print("=== First record - all fields ===")
    for k, v in posts[0].items():
        vs = str(v)[:80] + "..." if len(str(v)) > 80 else str(v)
        print(f"  {k}: {vs}")
else:
    print(f"Error: {result.error}")

---
## Test 4: Posts — Discover by Profiles Array

Discover posts from several profiles in a single request (one input row carrying
the whole array of profile URLs).

In [3]:
PROFILE_URLS = [
    "https://x.com/cnn",
    "https://x.com/BSCNews",

]

# Cap each profile to a few posts so the job stays small and fast while testing.
async with client:
    result = await client.scrape.x.posts_by_profiles_array(
        urls=PROFILE_URLS, limit_per_input=2, timeout=1000
    )

print(f"Success: {result.success} | Status: {result.status}")

if result.success and result.data:
    posts = result.data if isinstance(result.data, list) else [result.data]
    print(f"Discovered {len(posts)} post(s) across {len(PROFILE_URLS)} profiles")
    print()
    for i, p in enumerate(posts[:10]):
        text = p.get("description") or "(no caption - media/repost)"
        print(f"  {i+1}. @{p.get('user_posted', 'N/A')}: {str(text)[:70]}")
else:
    print(f"Error: {result.error}")

Success: True | Status: ready
Discovered 1 post(s) across 2 profiles

  1. @N/A: (no caption - media/repost)


---
## Test 5: Profiles — Collect by URL

Collect a profile by its URL. `max_number_of_posts` caps how many posts to pull.

In [ ]:
PROFILE_URL = "https://x.com/elonmusk"

async with client:
    result = await client.scrape.x.profiles(url=PROFILE_URL, max_number_of_posts=20)

print(f"Success: {result.success} | Status: {result.status}")

if result.success and result.data:
    data = result.data
    print(f"\n=== Profile Data ({len(data)} fields) ===")
    for key, value in data.items():
        val_str = str(value)[:80] + "..." if len(str(value)) > 80 else str(value)
        print(f"  {key}: {val_str}")
else:
    print(f"Error: {result.error}")

---
## Test 6: Profiles — Discover by User Name

Discover a profile by handle (no URL needed) — pass one handle or a list.

In [ ]:
USER_NAMES = ["elonmusk", "BillGates"]

async with client:
    result = await client.scrape.x.profiles_by_username(user_name=USER_NAMES)

print(f"Success: {result.success} | Status: {result.status}")

if result.success and result.data:
    profiles = result.data if isinstance(result.data, list) else [result.data]
    print(f"\nDiscovered {len(profiles)} profile(s):")
    for pr in profiles:
        print(f"  - {pr.get('name', 'N/A')} (@{pr.get('id', pr.get('user_name', 'N/A'))})")
else:
    print(f"Error: {result.error}")

---
## Test 7: Manual Job Control (Trigger / Status / Fetch)

For long-running operations, control the workflow manually. Every verb has a
matching `*_trigger` / `*_status` / `*_fetch` (here shown for posts).

In [ ]:
import asyncio

POST_URL = "https://x.com/CNN/status/1796673270344810776"

print(f"Manual workflow for: {POST_URL}\n")

async with client:
    # Step 1: trigger
    print("Step 1: Triggering scrape...")
    job = await client.scrape.x.posts_trigger(url=POST_URL)
    print(f"  Job ID: {job.snapshot_id}")

    # Step 2: poll
    print("\nStep 2: Polling for status...")
    while True:
        status = await job.status()
        print(f"  Status: {status}")
        if status == "ready":
            break
        elif status in ("error", "failed"):
            print("  Job failed!")
            break
        await asyncio.sleep(5)

    # Step 3: fetch
    if status == "ready":
        print("\nStep 3: Fetching results...")
        data = await job.fetch()
        item = data[0] if isinstance(data, list) else data
        print(f"  Got: {str(item)[:120]}")

---
## Sync usage

Every method is also available synchronously via `SyncBrightDataClient`
(`client.scrape.x.<method>(...)` with no `await`).

In [ ]:
from brightdata import SyncBrightDataClient

with SyncBrightDataClient(token=API_TOKEN) as sync_client:
    result = sync_client.scrape.x.posts(url="https://x.com/CNN/status/1796673270344810776")
    print(f"Success: {result.success} | Status: {result.status}")